# aw_07_b4 — Stage B4: P1 champion → PlayWorld SFT (Track B Phase-2, §5.1 / RQ1)

**Parent = P1 champion = B1v2** (`20260807-225109--b1-general-sft-v2--s42--e6e83b`,
sha256:747e5757...; §6 selection recorded in aw_06_b3). B3 (P1 DPO) was a null
transfer result, so the two-stage pipeline proceeds from the SFT champion.

**Design.** B4 is the RQ1 treatment arm: identical PlayWorld SFT budget to A1
(same data seed 1042, same recipe hyperparameters, 2 epochs full budget — NOT
the 200-step probe), differing from A1 in exactly one variable: initialization
(B1v2 P1 adapter vs base). Primary readout: `f_b4_analysis` B4 vs A1 on the
frozen suites (eval-ID + OOD splits), plus B4 vs A2 as the strongest Track-A
comparator.

Cell order: fetch champion → `a_b4_data` → `b_b4_train` → `c_b4_eval` →
`x09f_run_audit` → `f_b4_analysis`.

**Stage record (2026-08-13):** run `20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6`
(parent sha 747e5757..., output sha 378ba470..., train_loss profile healthy, 2ep/250 steps).
Eval run `20260813-000613--eval-playworld--s42--c49f3a`, freeze_fingerprint 3cdcbc30 MATCH,
truncation/runaway 0.0. **RQ1 primary: B4 ≫ A1 on ALL 10 suite×metric cells (p=.0001):**
eval-ID pass +.200 (.3867 vs .1867), template-OOD +.157, comp-OOD +.160, rule-OOD +.127,
adversarial +.250. B4 also ≫ A2 on all cells (p=.0001).

**⚠ OPEN AUDIT (x15): sft_fingerprint drift.** a_b4_data produced
sha256:2764e797... but the aw_05/aw_06 probe builds produced sha256:050f94b2...
(identical CLI/seed; prompt_fingerprint cc2aef0d... matched). The divergence is
confined to the SFT serialization. Run `scripts/x15_sft_data_diff.py` against the
frozen A1-era artifact before treating the RQ1 result as final: multiset-identical
⇒ caveat only; content-diverged ⇒ retrain B4 on the frozen artifact.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title fetch champion — materialize the B1v2 parent adapter + lineage sha
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b1v2_dir, b1v2_sha)


In [ ]:
# @title a_b4_data — deterministic PlayWorld train data + frozen suites (leakage-gated)
# Same generator/seed as A1 (seed 1042, 5 families x 400). Verify the manifest
# sft_fingerprint matches the A1-era value sha256:050f94b2... — any mismatch
# breaks the single-variable design and MUST stop the stage.
!python scripts/build_training_data.py \
  --seed 1042 --scenarios-per-family 400 --output-dir data/train

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_manifest fingerprint MUST equal sha256:3cdcbc30c99e492c... (G3 freeze)


In [ ]:
# @title b_b4_train — PlayWorld SFT from the B1v2 parent (full A1 budget)
!python scripts/run_experiment.py \
  --config configs/experiments/b4_playworld_sft_from_p1.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_run_id={B1V2_RUN_ID} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.revision=main \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title c_b4_eval — B4 adapter on the frozen suites (canonical profile)
B4_RUN_ID = ""  # <- from b_b4_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}
b4_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title x09f_run_audit — termination regression check on the B4 eval (CPU)
B4_EVAL = ""  # <- eval run id from c_b4_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B4_EVAL} --out runs/x09_run_audit_b4.json


In [ ]:
# @title f_b4_analysis — B4 vs A1 (RQ1 primary) and B4 vs A2
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"
A2_EVAL = ""  # <- canonical A2 eval run id (from aw_04_a2)

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B4_EVAL} --label-a b4-two-stage \
  --run-b runs/{A1_EVAL} --label-b a1-direct \
  --output runs/{B4_EVAL}/analysis_b4_vs_a1.json --hf-sync-repo m97j/aw-runs-b4

if A2_EVAL:
    !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL} --kind eval
    !python scripts/run_analysis.py \
      --run-a runs/{B4_EVAL} --label-a b4-two-stage \
      --run-b runs/{A2_EVAL} --label-b a2-dpo \
      --output runs/{B4_EVAL}/analysis_b4_vs_a2.json --hf-sync-repo m97j/aw-runs-b4


## Stage checklist — results in, one audit open (2026-08-13)
- [!] a_b4_data fingerprints: suites freeze 3cdcbc30 MATCH; **sft_fingerprint MISMATCH**
      (2764e797 vs probe-era 050f94b2; prompts cc2aef0d match) → x15 audit REQUIRED
- [x] b_b4_train lineage verified (parent sha 747e5757... = B1v2; output 378ba470...)
- [x] c_b4_eval recorded: ID .3867 / template .3567 / rule .3200 / comp .2833 / adv .8433
- [x] x09f: truncation_rate 0.0, runaway_rate 0.0 (prediction mean 255.6 chars)
- [x] RQ1 primary: B4 vs A1 — all 10 cells significant, p=.0001 (min delta +.127)
- [x] B4 vs A2 — all 10 cells significant, p=.0001
- [ ] **x15_sft_data_diff verdict recorded** (gates final acceptance of the above)
- [ ] x15 PASS ⇒ proceed to B5 (aw_08); FAIL ⇒ retrain B4 on frozen A1-era artifact
